Here's every column in your describe table:

| Column | What it answers | Example (`taxvaluedollarcnt`) |
|---|---|---|
| **`count`** | How many non-null values exist? Missing rows are excluded. | 77,578 out of 77,613 → 35 nulls |
| **`min`** | What's the smallest value? | $1,000 |
| **`max`** | What's the largest value? | $49,061,240 |
| **`mean`** | What's the average? (sum of all values / count) Sensitive to outliers. | ~$490K |
| **`std`** | How spread out are the values from the mean? Low = tightly clustered, high = widely spread. | ~$654K (very high → huge spread) |
| **`25%`** | The value below which 25% of data falls. (1st quartile / Q1) | ~$207K |
| **`50%`** | The middle value — **the median**. 50% of data is above, 50% below. Robust to outliers unlike mean. | ~$359K |
| **`75%`** | The value below which 75% of data falls. (3rd quartile / Q3) | ~$569K |
| **`skew`** | How asymmetric is the distribution? `0` = symmetric, `positive` = right-tail, `negative` = left-tail. | +31.5 (extreme right skew) |
| **`skewed`** | Your color-coded flag: **YES** if \|skew\| > 1, **moderate** if 0.5–1, **no** if < 0.5. | YES |

**Quick rules of thumb:**

- If `mean` >> `50%` → right-skewed (long tail of high values, like property prices)
- If `mean` << `50%` → left-skewed
- If `std` > `mean` → very high variance, likely has outliers
- `75% - 25%` = **IQR** (interquartile range) — the "middle 50%" spread, useful for outlier detection (anything beyond 1.5×IQR from Q1/Q3 is a potential outlier)

## Null value Heatmap

Here's how to read the missing-value heatmap:

### The Axes
- **X-axis** (bottom) — each column is one **feature** (all 55 columns)
- **Y-axis** (left) — each row is one **sample** (all 77,613 rows)

### The Colors
| Color | Meaning |
|---|---|
| **Dark/Blue** | Value **exists** (non-null) |
| **Yellow/Bright** | Value is **missing** (NaN) |

### How to Read Patterns

**Solid blue column** → that feature has almost no missing values (e.g., `bathroomcnt`, `latitude`)

**Solid yellow column** → that feature is almost entirely missing (e.g., `buildingclasstypeid` at 99.98% null)

**Mixed blue+yellow column** → partial missingness. The proportion of yellow tells you roughly how much is missing.

**Horizontal yellow bands** (rows that are yellow across many columns) → specific **samples** that are missing many features at once. These are your "high-null row" candidates to drop in notebook 03.

**Yellow columns that appear together** → features that tend to be missing for the **same rows**. For example, if `garagecarcnt` and `garagetotalsqft` are both yellow in the same rows, it means "if there's no garage data, both fields are missing" — they share a missingness pattern. This tells you the data is not missing randomly (MNAR — Missing Not At Random), which matters for imputation strategy.

### What to Look For in Your Zillow Heatmap

1. **Left side** — you should see several almost-entirely-yellow columns. Those are your 20 features with >90% missing that you already identified.

2. **Right side** — mostly solid blue columns. Those are the well-populated features like `latitude`, `longitude`, `taxvaluedollarcnt`.

3. **Block patterns** — look for groups of columns that go yellow together. The pool columns (`poolcnt`, `pooltypeid7`, `pooltypeid2`, `pooltypeid10`, `poolsizesum`) should show a similar yellow pattern because they're all missing for the same "no pool" properties. Same for garage columns.

4. **Scattered yellow dots** — random individual missing values, likely safe to impute with median/mode.

The heatmap is a quick visual confirmation of what your `missing_df` table already told you numerically — but it reveals **co-missingness patterns** that a simple percentage table cannot.